In [ ]:
// ==== Setup ====

import java.util.*;

// wall-clock time of r, in milliseconds
long millis(Runnable r) {
    System.gc();            // start from a clean heap: less noise
    long start = System.nanoTime();
    r.run();
    return (System.nanoTime() - start) / 1_000_000;
}

// test helpers
void check(boolean ok, String what) {
    System.out.println((ok ? "ok    " : "FAIL  ") + what);
}

# Trees

## From linear to hierarchical

- Lists, stacks, queues, deques: each element has **at most one** successor.
- **Tree**: a node can have **several** children.
- Examples: file system, organization chart, parse tree, family tree (descendants only).

## Terminology

- **root**: the only node with no parent; drawn at the top
- **edge**: a pair (parent, child)
- **path**: a sequence of nodes, each connected to the next by an edge

From the root to any node there is exactly **one** path. Consequence: a tree has no cycles.

## Leaves and internal nodes

- **leaf** (**external node**): a node with no children
- **internal node**: a node with at least one child

A list has exactly one leaf, its last node. A tree can have many.

## Subtree

- A node, together with all its descendants, forms a **subtree** rooted at that node.
- Remove the edge to its parent: the node becomes the root of its own (smaller) tree.
- A tree with `N` nodes has `N` subtrees, one per node.

## Parent, children, siblings

- **children** of a node: its direct successors
- **parent** of a node: its one direct predecessor (the root has none)
- **siblings**: nodes that share the same parent

## Ascendants and descendants

- **descendants** of a node: its children, their children, and so on
- **ascendants** of a node: its parent, its parent's parent, and so on, up to the root

A leaf has no descendants. The root has no ascendants, and every other node as a descendant.

## Level, depth, height

- **level** of a node: distance from the root. The root is at level `0`.
- **depth** of a node: number of ascendants (same count as the level). By analogy with height: `0` for the root; otherwise `1 + depth of its parent`.
- **height** of a node: `0` for a leaf; otherwise `1 + max(height of its children)`
- **depth/height of a tree**: level of its deepest node = height of the root = length of the longest root-leaf path
- Must the deepest node in a tree be a leaf?

## Ordered trees

- In general, the children of a node form a **set**: no order among them.
- A tree is **ordered** when the order of the children matters (e.g. left-to-right).

# Binary trees

## Definition

- Each node has **at most two** children, called **left** and **right**.
- We'll assume ordered binary trees, so left and right cannot be swapped.
- **left subtree**, **right subtree**: the subtrees rooted at the left and right child.

In [ ]:
class BTNode<E> {
    E data;
    BTNode<E> left, right;

    BTNode(E data) { this(data, null, null); }

    BTNode(E data, BTNode<E> left, BTNode<E> right) {
        this.data = data;
        this.left = left;
        this.right = right;
    }
}

In [ ]:
//      A
//    /   \
//   B      C
//  / \      \
// D   E      F

BTNode<String> d = new BTNode<>("D");
BTNode<String> e = new BTNode<>("E");
BTNode<String> b = new BTNode<>("B", d, e);
BTNode<String> f = new BTNode<>("F");
BTNode<String> c = new BTNode<>("C", null, f);
BTNode<String> root = new BTNode<>("A", b, c);

- Leaves of this tree? Internal nodes?
- Level of `D`? Depth of `C`?
- Height of `F`? Height of the whole tree?
- Siblings of `E`? Ascendants of `F`?

## How many nodes fit

- At level `l`: at most `2^l` nodes.
- `N` nodes, shortest possible tree: `⌊log₂ N⌋ + 1` levels.
- `N` nodes, tallest possible tree: `N` levels. Why?

## Full and complete binary trees

- **Full**: every node has `0` or `2` children, never exactly `1`.
- **Complete**: every level is full except possibly the last, which fills left to right.

- Is a complete binary tree always full? Is a full one always complete?

## General trees

- No bound on the number of children. The **degree** of a node: its number of children.
- A node with one fixed child field per possible child does not work: how many fields?

In [ ]:
// one child reference (the first child), one sibling reference (the next one):
// together they reach every child, in order
class TreeNode<E> {
    E data;
    TreeNode<E> firstChild, nextSibling;

    TreeNode(E data) { this(data, null, null); }

    TreeNode(E data, TreeNode<E> firstChild, TreeNode<E> nextSibling) {
        this.data = data;
        this.firstChild = firstChild;
        this.nextSibling = nextSibling;
    }
}

- Children of a node, in order: `firstChild`, `firstChild.nextSibling`, `firstChild.nextSibling.nextSibling`, ...
- A node with no children: `firstChild == null`.
- The last child among siblings: `nextSibling == null`.
- A `List<TreeNode<E>>` field would also work, and is closer to what real code tends to use. This representation is taught instead because it maps *any* general, ordered tree onto a binary tree: `firstChild` plays the role of left child, `nextSibling` the role of right child. The topology does not change; it is still a general tree, only now represented with binary-tree nodes.
- Once we have `size` and the traversals (below) for binary trees, which of them can run unchanged on a general tree, through this correspondence?

# Computing the size of a tree

## Size, recursively

- Trees are recursive: remove a node's edge to its parent, and what remains is still a tree.
- `size` of an empty tree: `0`. `size` of a node: `1 + size(left) + size(right)`.

In [ ]:
int size(BTNode<?> node) {
    if (node == null) return 0;
    return 1 + size(node.left) + size(node.right);
}

A real implementation keeps `size` as a field, updated on every insertion and removal. This recursive method is a template for other tree algorithms to come.

## Call trees

- A recursive call can make further recursive calls before it returns. Draw them as a tree: the first call is the root, each call it makes directly is a child, down to the base cases, which are the leaves.
- Mergesort and quicksort make up to two recursive calls each: their recursion is itself a binary tree, and its height is exactly what "how many levels" meant back then.
  - Mergesort always splits in half: height `⌊log₂ N⌋`, matching the shortest binary tree on `N` nodes.
  - Quicksort, bad pivot, sorted input: split `0` and `N - 1` every time. The recursion degenerates into a "stick": height `N - 1`, matching the tallest binary tree on `N` nodes.
- A function with a single recursive call (binary search, reversing a string): its call tree is a straight line, not really branching.
- `size(node)` makes two recursive calls per node: its call tree has the exact same shape as the tree it runs on. One call per node, one edge per child link.

In [ ]:
int sizeTraced(BTNode<?> node, int depth) {
    System.out.println("  ".repeat(depth) + (node == null ? "." : node.data));
    if (node == null) return 0;
    return 1 + sizeTraced(node.left, depth + 1) + sizeTraced(node.right, depth + 1);
}

check(sizeTraced(root, 0) == 6, "size, with a trace of every call");

- The indentation above traces the call tree directly: compare it to the picture of the data tree. While `size(node)` is paused waiting for `size(node.left)`, its stack frame is still open. At any such moment, the open frames are exactly the ascendants of the node currently running.
- How many calls does `size` make in total, for a tree of `N` nodes? (Count the calls on real nodes and the calls on `null` separately.)

## Size, iteratively

- Needs an explicit stack: one entry per node whose children have not yet been counted.

In [ ]:
int sizeIterative(BTNode<?> root) {
    if (root == null) return 0;
    Deque<BTNode<?>> stack = new ArrayDeque<>();
    stack.push(root);
    int count = 0;
    while (!stack.isEmpty()) {
        BTNode<?> current = stack.pop();
        count++;
        if (current.left != null) stack.push(current.left);
        if (current.right != null) stack.push(current.right);
    }
    return count;
}

In [ ]:
check(size(root) == 6, "size, recursive");
check(sizeIterative(root) == 6, "size, iterative");

- An empty stack means every node has been counted, or there is a bug.
- Replace the stack with a `Queue`: does it still count every node? What changes?

## A worked trace

Tree, written as `node(left, right)`:

`A(B(D, E(H, I)), C(F(, J), G(K, )))`

Trace of `sizeIterative`, one row per iteration:

| current | pushed | stack after |
|---|---|---|
| — | — | `[A]` |
| `A` | `B`, `C` | `[B, C]` |
| `C` | `F`, `G` | `[B, F, G]` |
| `G` | `K` | `[B, F, K]` |
| `K` | — | `[B, F]` |
| `F` | `J` | `[B, J]` |
| `J` | — | `[B]` |
| `B` | `D`, `E` | `[D, E]` |
| `E` | `H`, `I` | `[D, H, I]` |
| `I` | — | `[D, H]` |
| `H` | — | `[D]` |
| `D` | — | `[]` |

`count` reaches `11`.

In [ ]:
// A(B(D, E(H, I)), C(F(, J), G(K, )))
BTNode<String> tD = new BTNode<>("D");
BTNode<String> tH = new BTNode<>("H");
BTNode<String> tI = new BTNode<>("I");
BTNode<String> tE = new BTNode<>("E", tH, tI);
BTNode<String> tB = new BTNode<>("B", tD, tE);
BTNode<String> tJ = new BTNode<>("J");
BTNode<String> tF = new BTNode<>("F", null, tJ);
BTNode<String> tK = new BTNode<>("K");
BTNode<String> tG = new BTNode<>("G", tK, null);
BTNode<String> tC = new BTNode<>("C", tF, tG);
BTNode<String> tRoot = new BTNode<>("A", tB, tC);

check(size(tRoot) == 11, "size, recursive, matches the trace");
check(sizeIterative(tRoot) == 11, "size, iterative, matches the trace");

# Traversing a binary tree

Visit every node exactly once, to process its data. Which order is right depends on whether a node's data is needed before its children's, after them, or in between.

Running example: an expression tree, e.g. `(A + B) * C`, where leaves are operands and internal nodes are operators. Three standard orders: preorder, postorder, inorder.

## Preorder

The node, then left subtree, then right subtree.

- Needed whenever a node must be processed before its children: copying a tree, printing a hierarchy (list a directory, then its contents), serializing a tree to reconstruct it top-down (exercise 10 uses exactly this property).
- On the expression tree: the operator before its operands, i.e. prefix notation.

In [ ]:
void preorder(BTNode<String> node, List<String> out) {
    if (node == null) return;
    out.add(node.data);
    preorder(node.left, out);
    preorder(node.right, out);
}

## Postorder

Left subtree, then right subtree, then the node.

- Needed whenever a node depends on its children's results: `size`, height (both computed this way already), deleting a tree (free the children before the node itself, or the references needed to reach them are gone).
- On the expression tree: the operands before the operator, i.e. postfix notation. Evaluating a postfix expression with a stack is exactly this traversal.

In [ ]:
void postorder(BTNode<String> node, List<String> out) {
    if (node == null) return;
    postorder(node.left, out);
    postorder(node.right, out);
    out.add(node.data);
}

## Inorder

Left subtree, then the node, then right subtree.

- On the expression tree: operand, operator, operand, the usual left-to-right (infix) reading of an expression.
- The name itself will make more sense once we see binary search trees.

In [ ]:
void inorder(BTNode<String> node, List<String> out) {
    if (node == null) return;
    inorder(node.left, out);
    out.add(node.data);
    inorder(node.right, out);
}

In [ ]:
List<String> pre = new ArrayList<>(), post = new ArrayList<>(), in = new ArrayList<>();
preorder(root, pre);
postorder(root, post);
inorder(root, in);
System.out.println("preorder:  " + pre);
System.out.println("postorder: " + post);
System.out.println("inorder:   " + in);

check(pre.equals(List.of("A", "B", "D", "E", "C", "F")), "preorder");
check(post.equals(List.of("D", "E", "B", "F", "C", "A")), "postorder");
check(in.equals(List.of("D", "B", "E", "A", "C", "F")), "inorder");

## Iterative inorder

The recursive call stack is replaced by an explicit stack: push nodes while going left; when there is nowhere further left to go, pop one, process it, then go right.

- The explicit stack plays the same role as the call stack above: at any moment, it holds exactly the ascendants of `current`.

In [ ]:
List<String> inorderIterative(BTNode<String> root) {
    List<String> out = new ArrayList<>();
    Deque<BTNode<String>> stack = new ArrayDeque<>();
    BTNode<String> current = root;
    while (current != null || !stack.isEmpty()) {
        if (current != null) {
            stack.push(current);
            current = current.left;
        } else {
            current = stack.pop();
            out.add(current.data);
            current = current.right;
        }
    }
    return out;
}

In [ ]:
check(inorderIterative(root).equals(in), "iterative inorder matches recursive");

- Could the stack be replaced with a `Queue`? Trace a few steps: does the order still come out inorder?
- Write the iterative versions of preorder and postorder. Preorder is a small change to the code above. Postorder is not.

## Exercises

### 1. Full or complete?
Each tree below is binary, written as `node(left, right)`; `()` means no children.

- T1: `A(B(D,E), C(F,G))`
- T2: `A(B(D,E), C(F))`
- T3: `A(B(), C(D,E))`
- T4: `A(B(D), C(E,F))`

For each tree: is it full? Is it complete?

T1: full (every node has `0` or `2` children) and complete (both levels are as full as possible).

T2: complete (level 1 has `2` nodes, level 2 fills left to right: `D, E, F`), but not full (`C` has one child).

T3: full (`B` has `0` children, `C` has `2`), but not complete (level 2 is filled on the right, under `C`, while the leftmost slots under `B` are empty).

T4: neither (`B` has one child, and level 2 has a gap before a filled slot).

### 2. Level, depth, height
Tree: `A(B(D(I,J), E), C(F, G(H)))`.

- Level and depth of each node.
- Height of each node.
- Height of the tree. Depth of the tree.

Levels (= depths): `A`: 0. `B`, `C`: 1. `D`, `E`, `F`, `G`: 2. `I`, `J`, `H`: 3.

Heights, bottom-up: `I`, `J`, `E`, `F`, `H`: 0 (leaves). `D` = `1 + max(0, 0)` = 1. `G` = `1 + height(H)` = 1. `B` = `1 + max(height(D), height(E))` = `1 + max(1, 0)` = 2. `C` = `1 + max(height(F), height(G))` = `1 + max(0, 1)` = 2. `A` = `1 + max(2, 2)` = 3.

Height of the tree: `3`. Depth of the tree: level of the deepest nodes (`I`, `J`, `H`), also `3`.

### 3. A general tree, by code
Build this org chart with `TreeNode<String>`: `CEO` has children `CTO`, `CFO`; `CTO` has children `Dev1`, `Dev2`.

- What is the degree of `CEO`? Of `CFO`?
- `TreeNode` gives the children of a node a left-to-right order. Does an org chart need one?

In [ ]:
TreeNode<String> dev1 = new TreeNode<>("Dev1");
TreeNode<String> dev2 = new TreeNode<>("Dev2");
TreeNode<String> cto = new TreeNode<>("CTO", dev1, null);
dev1.nextSibling = dev2;

TreeNode<String> cfo = new TreeNode<>("CFO");
TreeNode<String> ceo = new TreeNode<>("CEO", cto, null);
cto.nextSibling = cfo;

Degree of `CEO`: `2` (`CTO`, `CFO`). Degree of `CFO`: `0`, a leaf.

An org chart usually does not care about the order of `CTO` vs. `CFO`: the structure imposes an order that the data does not need.

### 4. Counting nodes
A binary tree has height `h`.

- Fewest possible nodes? Which shape achieves it?
- Most possible nodes? Which shape achieves it?

A binary tree has `100` nodes.

- Smallest possible height?
- Largest possible height?

Height `h`: fewest nodes is `h + 1`, a tree where every node has exactly one child (a "stick"). Most nodes is `2^(h+1) - 1`, a tree that is full and complete at every level (a *perfect* tree).

`N = 100`: smallest height is `⌊log₂ 100⌋ = 6` (shortest tree has `⌊log₂ N⌋ + 1` levels, i.e. height `⌊log₂ N⌋`). Largest height is `99`, one child per node.

### 5. Size of a general tree
Write a recursive `size` for `TreeNode<E>`. A node holds a reference to its first child only, not to each child directly.

In [ ]:
int size(TreeNode<?> node) {
    if (node == null) return 0;
    int count = 1;
    for (TreeNode<?> child = node.firstChild; child != null; child = child.nextSibling)
        count += size(child);
    return count;
}

check(size(ceo) == 5, "CEO's org chart: CEO, CTO, CFO, Dev1, Dev2");

### 6. Trace the iterative inorder
Same tree as the size trace: `A(B(D, E(H, I)), C(F(, J), G(K, )))`.

Trace `inorderIterative` on it: one row per push or pop.

| action | node | stack after |
|---|---|---|
| push | `A` | `[A]` |
| push | `B` | `[A, B]` |
| push | `D` | `[A, B, D]` |
| pop → out | `D` | `[A, B]` |
| pop → out | `B` | `[A]` |
| push | `E` | `[A, E]` |
| push | `H` | `[A, E, H]` |
| pop → out | `H` | `[A, E]` |
| pop → out | `E` | `[A]` |
| push | `I` | `[A, I]` |
| pop → out | `I` | `[A]` |
| pop → out | `A` | `[]` |
| push | `C` | `[C]` |
| push | `F` | `[C, F]` |
| pop → out | `F` | `[C]` |
| push | `J` | `[C, J]` |
| pop → out | `J` | `[C]` |
| pop → out | `C` | `[]` |
| push | `G` | `[G]` |
| push | `K` | `[G, K]` |
| pop → out | `K` | `[G]` |
| pop → out | `G` | `[]` |

Result: `D, B, H, E, I, A, F, J, C, K, G`.

In [ ]:
check(inorderIterative(tRoot).equals(List.of("D", "B", "H", "E", "I", "A", "F", "J", "C", "K", "G")), "trace");

### 7. Iterative preorder and postorder
Write `preorderIterative` and `postorderIterative` for `BTNode<String>`.
- Preorder: a small change to `inorderIterative`.
- Postorder: harder. Hint: a traversal that visits node, then right, then left, reversed, gives postorder.

In [ ]:
List<String> preorderIterative(BTNode<String> root) {
    List<String> out = new ArrayList<>();
    if (root == null) return out;
    Deque<BTNode<String>> stack = new ArrayDeque<>();
    stack.push(root);
    while (!stack.isEmpty()) {
        BTNode<String> node = stack.pop();
        out.add(node.data);
        if (node.right != null) stack.push(node.right);   // pushed first: popped after the left child
        if (node.left != null) stack.push(node.left);
    }
    return out;
}

List<String> postorderIterative(BTNode<String> root) {
    Deque<BTNode<String>> stack = new ArrayDeque<>();
    LinkedList<String> out = new LinkedList<>();     // addFirst: builds the result in reverse
    if (root == null) return out;
    stack.push(root);
    while (!stack.isEmpty()) {
        BTNode<String> node = stack.pop();
        out.addFirst(node.data);
        if (node.left != null) stack.push(node.left);
        if (node.right != null) stack.push(node.right);
    }
    return out;
}

check(preorderIterative(tRoot).equals(List.of("A", "B", "D", "E", "H", "I", "C", "F", "J", "G", "K")), "preorder");
check(postorderIterative(tRoot).equals(List.of("D", "H", "I", "E", "B", "J", "F", "K", "G", "C", "A")), "postorder");

### 8. Inorder with a queue
In `inorderIterative`, replace the `Deque` used as a stack with one used as a `Queue` (`poll` instead of `pop`). Does the result still come out inorder?

No. The algorithm relies on the stack to undo exactly the descent that was just made, in reverse: the last node pushed is the first one that must come back to continue inorder. A queue returns nodes in the order they were pushed, not in reverse, so it loses track of which ancestor to return to. (This is the same reason the recursive version uses the call stack, not some other structure.)

### 9. Specify the traversals
Same tree again: `A(B(D, E(H, I)), C(F(, J), G(K, )))`.

For each of inorder, preorder, postorder:
- Write out the full traversal.
- Which node is visited 5th?
- Which node is visited right after `I`?

| | traversal | 5th node | node after `I` |
|---|---|---|---|
| inorder | `D, B, H, E, I, A, F, J, C, K, G` | `I` | `A` |
| preorder | `A, B, D, E, H, I, C, F, J, G, K` | `H` | `C` |
| postorder | `D, H, I, E, B, J, F, K, G, C, A` | `B` | `E` |

### 10. Tree reconstruction
A binary tree's preorder is `A, B, D, E, H, I, C, F, J, G, K`; its inorder is `D, B, H, E, I, A, F, J, C, K, G`. Reconstruct the tree.

- The first node of a preorder is always the root.
- Find that root in the inorder: everything before it is the left subtree, everything after is the right subtree.
- The sizes of those two parts split the rest of the preorder into the preorders of the left and right subtrees. Recurse.

Root `A`, splitting inorder into `D, B, H, E, I` (left) and `F, J, C, K, G` (right), and preorder into `B, D, E, H, I` (left) and `C, F, J, G, K` (right).

Recursing the same way on each side gives back exactly the tree from the worked trace: `A(B(D, E(H, I)), C(F(, J), G(K, )))`.